# **Fine-Tuning LLaMA-3.2-3B with DPO and PEFT**

# Introduction
This notebook offers a detailed guide for fine-tuning a LLaMA-based language model using the Unsloth library. It starts with setting up the required environment and dependencies, followed by loading a pre-trained model, with the option to apply 4-bit quantization for better memory efficiency. The workflow includes implementing Parameter-Efficient Fine-Tuning (PEFT) using LoRA, preparing a preference-based dataset, and configuring the Direct Preference Optimization (DPO) trainer for model training. Additionally, the notebook demonstrates how to perform inference, stream text generation in real-time, and save the fine-tuned model in multiple formats, making it suitable for various deployment environments.

# Setup and Installation

In [ ]:
%%capture
# Install pip3-autoremove to easily remove old versions of packages
!pip install pip3-autoremove

# Remove existing versions of torch, torchvision, and torchaudio
!pip-autoremove torch torchvision torchaudio -y

# Install the specified versions of torch, torchvision, and torchaudio with CUDA 12.1 support
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121

# Install the latest stable version of Unsloth
!pip install unsloth

# Uninstall the current version of Unsloth and install the latest nightly build from GitHub
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

# Upgrade transformers package to the latest version
!pip install --upgrade --no-cache-dir transformers

# Loading the Language Model

In [2]:
from unsloth import FastLanguageModel
import torch

# Set the maximum sequence length
max_seq_length = 2048  # Choose any value; RoPE scaling is supported internally

# Data type configuration for model precision (auto-detection by default)
dtype = None  # None will auto-detect, use float16 for Tesla T4/V100, or bfloat16 for Ampere+ GPUs

# Use 4-bit quantization to reduce memory usage (set to False to disable)
load_in_4bit = True  

# Load the model and tokenizer from pretrained weights
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",  # Optionally choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

# Applying PEFT (Parameter-Efficient Fine-Tuning)

In [3]:
# Apply PEFT (Parameter-Efficient Fine-Tuning) to the model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Choose any value > 0! Suggested values: 8, 16, 32, 64, 128
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    ],  # List of modules to apply PEFT to
    lora_alpha=16,  # Scaling factor for LoRA (low-rank adaptation)
    lora_dropout=0,  # Dropout rate for LoRA; optimized for 0
    bias="none",  # Bias handling, "none" is optimized for this configuration
    use_gradient_checkpointing="unsloth",  # Use "unsloth" for reduced VRAM usage and longer contexts
    random_state=3407,  # Random seed for reproducibility
    use_rslora=False,  # Set to True if using rank stabilized LoRA
    loftq_config=None  # LoftQ configuration (set to None for no use)
)

Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Preparing the Dataset

In [4]:
# Define the Alpaca prompt format with placeholders for instruction, input, and response
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Get the EOS token from the tokenizer
EOS_TOKEN = tokenizer.eos_token  # Ensure the EOS token is included

# Function to format the dataset sample
# This function modifies the prompt, chosen response, and rejected response
def format_prompt(sample):
    instruction = "You are an AI assistant. You will be given a task. You must generate a correct answer."
    input_text = sample["prompt"]
    accepted = sample["chosen"]
    rejected = sample["rejected"]

    # Format the sample's prompt and append EOS token to both the accepted and rejected responses
    sample["prompt"] = alpaca_prompt.format(instruction, input_text, "")
    sample["chosen"] = accepted + EOS_TOKEN
    sample["rejected"] = rejected + EOS_TOKEN
    return sample

# Load the dataset
from datasets import load_dataset
dataset = load_dataset("ogbrandt/gpt4_preference_rlaif")["train"]

# Apply the format_prompt function to each sample in the dataset
dataset = dataset.map(format_prompt)

README.md:   0%|          | 0.00/548 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/327k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/520 [00:00<?, ? examples/s]

Map:   0%|          | 0/520 [00:00<?, ? examples/s]

In [5]:
import pprint

# Select a specific sample (row) from the dataset
row = dataset[1]

# Print the instruction, accepted, and rejected samples with clear headings
print('INSTRUCTION: ' + '=' * 50)
pprint.pprint(row["prompt"])  # Display the formatted prompt (instruction)

print('ACCEPTED: ' + '=' * 50)
pprint.pprint(row["chosen"])  # Display the accepted response

print('REJECTED: ' + '=' * 50)
pprint.pprint(row["rejected"])  # Display the rejected response

INSTRUCTION: ==================================================
('Below is an instruction that describes a task, paired with an input that '
 'provides further context. Write a response that appropriately completes the '
 'request.\n'
 '\n'
 '### Instruction:\n'
 'You are an AI assistant. You will be given a task. You must generate a '
 'correct answer.\n'
 '\n'
 '### Input:\n'
 'What mental exercises can facilitate a quicker return to sport post-injury?\n'
 '\n'
 '### Response:\n')
ACCEPTED: ==================================================
('To facilitate a quicker return to sport post-injury, mental exercises can '
 'play a significant role. These exercises can help improve focus, '
 'concentration, and self-confidence. Encourage your client to practice '
 'visualization techniques, such as imagining themselves in competitive '
 'situations and overcoming obstacles. Additionally, mindfulness exercises, '
 'like deep breathing and meditation, can help reduce stress and anxiety, '
 '

# Configuring the DPO Trainer

In [6]:
# Enable reward modeling stats using PatchDPOTrainer
from unsloth import PatchDPOTrainer
PatchDPOTrainer()  # Initializes PatchDPOTrainer for reward modeling

from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

# Configure and initialize the DPOTrainer with the given training settings
dpo_trainer = DPOTrainer(
    model=model,  # The model to be fine-tuned
    ref_model=None,  # No reference model used in this case
    args=DPOConfig(
        per_device_train_batch_size=2,  # Batch size per device
        gradient_accumulation_steps=4,   # Accumulate gradients over 4 steps
        warmup_ratio=0.1,                # Warmup ratio for learning rate
        num_train_epochs=1,              # Number of training epochs
        learning_rate=5e-6,              # Learning rate for optimization
        fp16=not is_bfloat16_supported(),  # Use FP16 if bfloat16 is not supported
        bf16=is_bfloat16_supported(),     # Use BF16 if supported by the hardware
        logging_steps=1,                  # Log every step
        optim="adamw_8bit",               # Use the 8-bit AdamW optimizer
        weight_decay=0.0,                 # No weight decay
        lr_scheduler_type="linear",      # Use linear learning rate scheduler
        seed=42,                          # Set random seed for reproducibility
        output_dir="outputs",            # Directory to save model outputs
        report_to="none",                # Report to none (can be adjusted for WandB or other services)
    ),
    beta=0.1,                          # Beta value used in the DPO algorithm
    train_dataset=dataset,              # Training dataset
    tokenizer=tokenizer,                # Tokenizer for text processing
    max_length=1024,                    # Maximum sequence length for input
    max_prompt_length=512,              # Maximum length of the prompt
)

Extracting prompt in train dataset (num_proc=4):   0%|          | 0/520 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=4):   0%|          | 0/520 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=4):   0%|          | 0/520 [00:00<?, ? examples/s]

# Starting Training

In [7]:
dpo_trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 520 | Num Epochs = 1 | Total steps = 32
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856/3,000,000,000 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss,aux_loss
1,0.693100,0.000000,0.000000,0.000000,0.000000,-145.716446,-132.331726,0.056684,0.155723,0,0,0,0
2,0.693100,0.000000,0.000000,0.000000,0.000000,-137.152405,-124.593384,0.001290,-0.020348,No Log,No Log,No Log,No Log
3,0.692200,-0.001085,-0.003018,0.687500,0.001933,-131.706207,-122.488068,0.113477,0.272781,No Log,No Log,No Log,No Log
4,0.693500,0.000360,0.001045,0.375000,-0.000685,-141.918274,-122.024582,0.060560,0.080459,No Log,No Log,No Log,No Log
5,0.693500,-0.000932,-0.000203,0.375000,-0.000729,-146.191574,-143.525772,0.038924,0.051886,No Log,No Log,No Log,No Log
6,0.693100,-0.001459,-0.001604,0.625000,0.000145,-139.361877,-127.193604,-0.132069,-0.075112,No Log,No Log,No Log,No Log
7,0.692100,-0.001005,-0.003046,0.625000,0.002042,-150.215820,-119.839554,0.121112,0.070330,No Log,No Log,No Log,No Log
8,0.693700,-0.002177,-0.001084,0.437500,-0.001093,-129.282806,-129.011612,0.129257,0.177085,No Log,No Log,No Log,No Log
9,0.692900,-0.005483,-0.006047,0.500000,0.000564,-135.992722,-112.253525,0.056880,0.004139,No Log,No Log,No Log,No Log
10,0.691400,-0.002626,-0.006138,0.625000,0.003512,-146.358643,-134.762360,0.137117,0.141118,No Log,No Log,No Log,No Log


TrainOutput(global_step=32, training_loss=0.6875784154981375, metrics={'train_runtime': 389.7725, 'train_samples_per_second': 1.334, 'train_steps_per_second': 0.082, 'total_flos': 0.0, 'train_loss': 0.6875784154981375, 'epoch': 0.9846153846153847})

# Inference: Generating Text

In [8]:
FastLanguageModel.for_inference(model)

# Prepare the input prompt for generation using the tokenizer
inputs = tokenizer(
    [
        alpaca_prompt.format(
            "Continue the fibonacci sequence.",  # Instruction: continue the Fibonacci sequence
            "1, 1, 2, 3, 5, 8",                   # Input: starting sequence
            "",                                   # Output: leave blank for generation
        )
    ], 
    return_tensors="pt"  # Convert inputs to PyTorch tensors
).to("cuda")  # Move the tensors to GPU for faster processing

# Generate the output sequence
outputs = model.generate(
    **inputs,               # Pass the prepared input tensors
    max_new_tokens=64,      # Generate up to 64 new tokens
    use_cache=True          # Enable cache for faster inference
)

# Decode the generated tokens into human-readable text
generated_text = tokenizer.batch_decode(outputs)
print(generated_text)  # Output the generated sequence

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nContinue the fibonacci sequence.\n\n### Input:\n1, 1, 2, 3, 5, 8\n\n### Response:\n11, 89, 144, 233, 377, 610, 985, 1597, 2584, 4181, 6765, 10946, 17711, 28657, 46368, 75025, 121393, 196418,']


# Inferencing with Streaming

In [9]:
FastLanguageModel.for_inference(model)

# Prepare the input prompt for generation using the tokenizer
inputs = tokenizer(
    [
        alpaca_prompt.format(
            "Continue the fibonacci sequence.",  # Instruction: continue the Fibonacci sequence
            "1, 1, 2, 3, 5, 8",                   # Input: current sequence
            "",                                   # Output: leave blank for generation
        )
    ], 
    return_tensors="pt"  # Convert the inputs into PyTorch tensors
).to("cuda")  # Move the tensors to GPU for faster processing

# Import the TextStreamer class to stream the generated text
from transformers import TextStreamer

# Initialize the TextStreamer with the tokenizer for streaming output
text_streamer = TextStreamer(tokenizer)

# Generate the output sequence with streaming enabled, and specify a max length of 128 new tokens
_ = model.generate(
    **inputs,                  # Pass the prepared input tensors
    streamer=text_streamer,     # Stream the output as it's generated
    max_new_tokens=128         # Generate up to 128 new tokens
)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Continue the fibonacci sequence.

### Input:
1, 1, 2, 3, 5, 8

### Response:
13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765, 10946, 17711, 28657, 46368, 75025, 121393, 196418, 317811, 514229, 832040, 1346269, 2178309, 3524578, 5702887, 9227465, 14930352, 24157817, 39088169, 632459


# Saving the Model Locally

In [10]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

# Saving the Model for vllm

In [11]:
# Saving the model in different formats for VLLM optimization
# Options include saving as merged 16-bit (float16), merged 4-bit (int4), or just LoRA adapters.

# Save the model in merged 16-bit (float16) format
# This option is commented out (False) to avoid saving during this run.
if False:
    model.save_pretrained_merged(
        "model",                   # Save the model to the 'model' directory
        tokenizer,                 # Save the tokenizer as well
        save_method="merged_16bit" # Choose merged 16-bit for float16 format
    )

# Save the model in merged 4-bit (int4) format
# This option is also commented out (False).
if False:
    model.save_pretrained_merged(
        "model",                   # Save the model to the 'model' directory
        tokenizer,                 # Save the tokenizer as well
        save_method="merged_4bit"  # Choose merged 4-bit for int4 format
    )

# Save only the LoRA adapters, without the full model
# This option is commented out (False).
if False:
    model.save_pretrained_merged(
        "model",                   # Save the model to the 'model' directory
        tokenizer,                 # Save the tokenizer as well
        save_method="lora"         # Save only the LoRA adapters
    )

# Saving the Model as GGUF (Ollama, llama.cpp)

In [12]:
# GGUF / Ollama / llama.cpp Conversion
# We now support saving models to GGUF and llama.cpp formats natively. 
# By default, we save to q8_0 format, but other quantization methods like q4_k_m are also supported.

# Save the model in 8-bit (Q8_0) format for GGUF or llama.cpp
# This option is currently disabled (False).
if False:
    model.save_pretrained_gguf(
        "model",            # Save the model to the 'model' directory
        tokenizer           # Save the tokenizer along with the model
    )

# Save the model in 16-bit (f16) format for GGUF or llama.cpp
# This option is also disabled (False).
if False:
    model.save_pretrained_gguf(
        "model",                   # Save the model to the 'model' directory
        tokenizer,                 # Save the tokenizer along with the model
        quantization_method="f16"  # Use 16-bit floating point (f16) quantization
    )

# Save the model in q4_k_m quantization method for GGUF or llama.cpp
# This option is also disabled (False).
if False:
    model.save_pretrained_gguf(
        "model",                   # Save the model to the 'model' directory
        tokenizer,                 # Save the tokenizer along with the model
        quantization_method="q4_k_m"  # Use q4_k_m quantization method
    )

# Conclusion
By following this notebook, users can effectively fine-tune a LLaMA language model for specific tasks and preferences using Unsloth and DPO. The step-by-step approach optimizes both training performance and memory usage, enabling efficient model fine-tuning. Additionally, the flexible saving options allow seamless integration into various deployment environments. This workflow empowers developers and researchers to efficiently customize and deploy powerful language models across different platforms, enhancing their ability to create tailored AI solutions.